# Film selection: five-decade check

Read one TMDB results page per decade, from 1950 to 1999. Sort by vote count to see familiar films first. This is a small coverage check, not the final sample.

Start Jupyter in the project root. Run top to bottom. Saved responses are used by default. Live requests read the token from the local `.env` file. Never print that file or the request headers.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import json
import time
import hashlib

folder = Path.cwd()
live = False
decades = range(1950, 2000, 10)
base = folder / "01_raw/tmdb_candidates"
latest = base / "latest_run.txt"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") if live else latest.read_text().strip()
run = base / run_id
run.mkdir(parents=True, exist_ok=True)
headers = {"User-Agent": "ADS206-film-study/0.1 (educational)", "Accept": "application/json"}
if live:
    settings = {}
    for line in (folder / ".env").read_text(encoding="utf-8-sig").splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            settings[key.strip()] = value.strip().strip("\"'")
    token = settings.get("TMDB_READ_TOKEN", "")
    assert token and not any(char.isspace() for char in token), "Check the local token field."
    headers["Authorization"] = "Bearer " + token
print("Mode:", "live requests" if live else "saved responses")

Mode: saved responses


## 1. Request and save

`requests.get` sends the web request. Keep each response and its log. Pause between requests; retry temporary failures up to three times. Headers and the token are never logged.

In [3]:
def fetch(params, name):
    if not live:
        return json.loads((run / name).read_bytes())
    for attempt in range(3):
        reply = None
        clock = time.perf_counter()
        log = {"time_utc": datetime.now(timezone.utc).isoformat(),
               "url": "https://api.themoviedb.org/3/discover/movie", "params": params,
               "retry_count": attempt, "collector": "AI-assisted candidate check"}
        try:
            reply = requests.get(log["url"], params=params, headers=headers, timeout=30)
            assert token.encode() not in reply.content, "Unexpected credential in response; not saved."
            tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
            file = f"{Path(name).stem}_{tag}.json"
            (run / file).write_bytes(reply.content)
            log.update(url=reply.url, status=reply.status_code, file=file,
                       redirects=[r.status_code for r in reply.history],
                       content_type=reply.headers.get("Content-Type"),
                       retry_after=reply.headers.get("Retry-After"),
                       sha256=hashlib.sha256(reply.content).hexdigest())
        except requests.RequestException as error:
            log["error"] = type(error).__name__
        log["seconds"] = round(time.perf_counter() - clock, 3)
        with (run / "requests.jsonl").open("a", encoding="utf-8") as file:
            file.write(json.dumps(log) + "\n")
        time.sleep(2)
        if reply is not None and reply.ok:
            (run / name).write_bytes(reply.content)
            return reply.json()
        if reply is not None and reply.status_code != 429 and reply.status_code < 500:
            raise RuntimeError(f"HTTP {reply.status_code}. See the request log.")
        if attempt == 2:
            raise ConnectionError("Request failed. See requests.jsonl.")
        delay = 2 ** attempt
        if reply is not None and reply.headers.get("Retry-After"):
            wait = reply.headers["Retry-After"]
            if not wait.isdigit() or int(wait) > 60:
                raise RuntimeError("Stop and review Retry-After before more requests.")
            delay = max(delay, int(wait))
        time.sleep(delay)

## 2. Read one page per decade

Use primary release dates and `vote_count.desc`. Keep all results on page 1; do not add a rating, vote-count, country or genre cutoff yet. English labels do not restrict the films to English-speaking countries.

The API's total result count describes this query. It is not a count of eligible classics, full-length films or available Cinemetrics measurements.

In this saved run, all five responses reported 20,001 results and 1,001 pages. The true totals were not verified. Only the returned records count as collected data.

In [4]:
records, checks = [], []
for decade in decades:
    params = {"primary_release_date.gte": f"{decade}-01-01",
              "primary_release_date.lte": f"{decade + 9}-12-31",
              "sort_by": "vote_count.desc", "page": 1, "language": "en-US",
              "include_adult": "false", "include_video": "false"}
    data = fetch(params, f"discover_{decade}.json")
    assert data["page"] == 1 and isinstance(data["results"], list)
    rows = data["results"]
    assert len({row["id"] for row in rows}) == len(rows), "Repeated ID on this page."
    assert all(decade <= int(row["release_date"][:4]) < decade + 10 for row in rows), "Check dates."
    assert all(isinstance(row["vote_count"], int) and isinstance(row["vote_average"], (int, float))
               for row in rows), "Check rating fields."
    assert all(a["vote_count"] >= b["vote_count"] for a, b in zip(rows, rows[1:])), "Check sorting."
    checks.append({"decade": f"{decade}s", "page_read": 1, "returned": len(rows),
                   "api_reported_total": data["total_results"], "api_reported_pages": data["total_pages"],
                   "blank_overviews": sum(not row.get("overview", "").strip() for row in rows)})
    for rank, row in enumerate(rows, 1):
        records.append({"tmdb_id": row["id"], "title": row["title"],
            "original_title": row["original_title"], "release_date": row["release_date"],
            "decade": f"{decade}s", "page": 1, "rank_on_page": rank,
            "rating": row["vote_average"], "vote_count": row["vote_count"],
            "original_language": row["original_language"], "genre_ids": row["genre_ids"],
            "overview": row.get("overview", ""), "selected_for_main_analysis": False,
            "tmdb_url": f"https://www.themoviedb.org/movie/{row['id']}"})
    print(f"{decade}s: read {len(rows)} candidates")
assert len({row["tmdb_id"] for row in records}) == len(records), "Repeated ID across decades."
view = pd.DataFrame(records)
pd.DataFrame(checks)[["decade", "page_read", "returned", "blank_overviews"]]

1950s: read 20 candidates
1960s: read 20 candidates
1970s: read 20 candidates
1980s: read 20 candidates
1990s: read 20 candidates


,decade,page_read,returned,blank_overviews
0,1950s,1,20,0
1,1960s,1,20,0
2,1970s,1,20,0
3,1980s,1,20,0
4,1990s,1,20,0


## 3. Save and review

Ratings are TMDB audience scores. Vote counts count ratings, not viewers. A title on this page is still a candidate: its Cinemetrics match and full-film boundaries have not been checked.

Before collecting more pages, choose and record common selection rules. More candidates will not remove the bias from using only the most-voted films.

In [5]:
result = {"stage": "one-page coverage check; final sample pending", "run_id": run_id,
          "source": "TMDB discover/movie", "sort_by": "vote_count.desc",
          "rating_cutoff": None, "vote_count_cutoff": None,
          "pages_per_decade": 1, "checks": checks, "records": records}
if live:
    body = json.dumps(result, ensure_ascii=False, indent=2)
    (run / "candidates.json").write_text(body, encoding="utf-8")
    (folder / "02_audit/tmdb_candidates.json").write_text(body, encoding="utf-8")
    latest.write_text(run_id, encoding="utf-8")
view[["decade", "title", "release_date", "rating", "vote_count", "original_language"]]

,decade,title,release_date,rating,vote_count,original_language
0,1950s,12 Angry Men,1957-04-10,8.577,10579,en
1,1950s,Rear Window,1954-08-01,8.333,7487,en
2,1950s,Cinderella,1950-02-22,7.057,7347,en
3,1950s,Vertigo,1958-05-28,8.131,6613,en
4,1950s,Alice in Wonderland,1951-08-09,7.197,6465,en
...,...,...,...,...,...,...
95,1990s,Léon: The Professional,1994-09-14,8.289,16777,fr
96,1990s,Star Wars: Episode I - The Phantom Menace,1999-05-19,6.582,16412,en
97,1990s,Reservoir Dogs,1992-09-02,8.109,16017,en
98,1990s,Toy Story 2,1999-10-30,7.624,15576,en


## Credits

<a href="https://www.themoviedb.org"><img src="https://www.themoviedb.org/assets/2/v4/logos/v2/blue_short-8e7b30f73a4020692ccca9c88bafe5dcb6f8a62a4c6bc55cd9ba82bb2cd95f6c.svg" width="120" alt="TMDB"></a>

This product uses the TMDB API but is not endorsed or certified by TMDB.

[TMDB API documentation](https://developer.themoviedb.org/reference/discover-movie) | [TMDB attribution](https://www.themoviedb.org/about/logos-attribution)

Codex assisted with acquisition code and replay checks. Final selection rules and interpretation need student review.